In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

1. LOAD DATA

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

2. PREPROCESSING & FEATURE ENGINEERING

In [3]:
def preprocess_data(df):
    # Drop ID as it must not be used for prediction per rules
    df_copy = df.drop(['id'], axis=1, errors='ignore').copy()

    # --- Ordinal Encoding (Mapping hierarchies manually) ---
    income_dict = {'<=2L': 0, '2L-5L': 1, '5L-10L': 2, 'More than 10L': 3}
    qual_dict = {'Others': 0, 'High School': 1, 'Bachelor': 2, 'Master': 3}
    policy_type_dict = {'Silver': 0, 'Gold': 1, 'Platinum': 2}
    num_policy_dict = {'1': 1, 'More than 1:': 2} # 'More than 1' is often the key indicator

    df_copy['income'] = df_copy['income'].map(income_dict)
    df_copy['qualification'] = df_copy['qualification'].map(qual_dict)
    df_copy['type_of_policy'] = df_copy['type_of_policy'].map(policy_type_dict)
    
    # Cleaning 'num_policies' column (Handling the "More than 1" string)
    df_copy['num_policies'] = df_copy['num_policies'].apply(lambda x: 2 if 'More' in str(x) else 1)

    # --- Label Encoding for remaining nominal categories ---
    le = LabelEncoder()
    for col in ['gender', 'area', 'policy', 'marital_status']:
        if col in df_copy.columns:
            df_copy[col] = le.fit_transform(df_copy[col].astype(str))

    # --- Feature Engineering ---
    # Interaction between Claim Amount and Vintage (Years with company)
    df_copy['claim_to_vintage_ratio'] = df_copy['claim_amount'] / (df_copy['vintage'] + 1)
    
    # Policy complexity (Number of policies * Type Tier)
    df_copy['policy_score'] = df_copy['num_policies'] * (df_copy['type_of_policy'] + 1)

    return df_copy

In [4]:
# Apply preprocessing
X_train_full = preprocess_data(train)
X_test = preprocess_data(test)

In [5]:
# Separate Target
y = X_train_full['cltv']
X = X_train_full.drop(['cltv'], axis=1)

3. MODEL SELECTION & HYPERPARAMETER TUNING

In [6]:
model = XGBRegressor(
    n_estimators=1000, 
    learning_rate=0.03, 
    max_depth=5, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    n_jobs=-1, 
    random_state=42,
    objective='reg:squarederror'
)

4. CROSS-VALIDATION

In [7]:
# CROSS-VALIDATION (Private Leaderboard Security)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')

print("-" * 30)
print(f"Mean R2 Score: {np.mean(cv_scores):.4f}")
print(f"Fold Scores: {cv_scores}")
print("-" * 30)

------------------------------
Mean R2 Score: 0.1503
Fold Scores: [0.14700264 0.14877522 0.1471265  0.15760809 0.15093446]
------------------------------


5. FINAL TRAINING & SUBMISSION

In [8]:
# Fit on the entire training set
model.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [9]:
# Predict on test data
final_predictions = model.predict(X_test)

In [10]:
# Create submission file
submission = pd.DataFrame({
    'id': test['id'],
    'cltv': final_predictions
})

In [11]:
submission.to_csv('final_cltv_submission.csv', index=False)
print("Submission file saved: final_cltv_submission.csv")

Submission file saved: final_cltv_submission.csv


6. MODEL INTERPRETABILITY

In [12]:
importances = pd.Series(model.feature_importances_, index=X.columns)
print("--- Top Predictors of CLTV: ---\n")
print(importances.sort_values(ascending=False).head(5))

--- Top Predictors of CLTV: ---

num_policies      0.618813
policy_score      0.126942
area              0.054115
type_of_policy    0.046797
claim_amount      0.027373
dtype: float32
